In [1]:
import requests
import json
import pandas as pd
import time
from datetime import datetime
import glob

from google.colab import userdata
API_KEY = userdata.get('DATAJUD_KEY')

URL = "https://api-publica.datajud.cnj.jus.br/api_publica_trf1/_search"
HEADERS = {"Authorization": API_KEY, "Content-Type": "application/json"}

In [2]:
def buscar_periodo(ano):
    gte = f"{ano}0101000000"
    lte = f"{ano}1231235959"

    query = {
      "size": 10000,
      "_source": ["numeroProcesso", "classe.nome", "dataAjuizamento", "assuntos", "orgaoJulgador.nome"],
      "query": {
        "bool": {
          "must": [
            { "terms": { "assuntos.codigo": [10110, 9994, 10116, 11828, 15302, 15301, 10114, 10113, 10119, 11822, 15008, 15300, 11830, 11825, 11829, 11824, 11823, 10112, 10111, 11862, 10115, 10118, 11827, 11826, 10438, 11869, 10396] } }
          ],
          "filter": [
            { "range": { "dataAjuizamento": { "gte": gte, "lte": lte } } }
          ]
        }
      },
      "sort": [{ "dataAjuizamento": "asc" }]
    }

    try:
        response = requests.post(URL, headers=HEADERS, json=query, timeout=60)
        return response.json()
    except Exception as e:
        print(f"Erro no ano {ano}: {e}")
        return None

In [3]:
def salvar_csv(dados, ano):
    if not dados or 'hits' not in dados or 'hits' not in dados['hits']:
        print(f"Aviso: Nenhum dado válido para o ano {ano}")
        return

    hits = dados['hits']['hits']
    total = dados['hits']['total']['value']
    print(f"Total encontrado em {ano}: {total}")

    processos_lista = []
    for h in hits:
        source = h.get('_source', {})

        # Tratamento de Assuntos
        lista_assuntos = source.get('assuntos', [])
        if lista_assuntos:
            nomes = [str(a.get('nome', a.get('codigo', 'N/A'))) for a in lista_assuntos if isinstance(a, dict)]
            assuntos_nomes = ", ".join(nomes)
        else:
            assuntos_nomes = "Assunto não informado"

        # Apóstrofo antes do número para tratar como texto
        numero_processo = source.get("numeroProcesso", "N/A")
        if numero_processo != "N/A":
            numero_processo = f"'{numero_processo}"

        processos_lista.append({
            "numero": numero_processo,
            "classe": source.get("classe", {}).get("nome", "N/A") if isinstance(source.get("classe"), dict) else "N/A",
            "data": source.get("dataAjuizamento", "N/A"),
            "orgao": source.get("orgaoJulgador", {}).get("nome", "N/A") if isinstance(source.get("orgaoJulgador"), dict) else "N/A",
            "assuntos": assuntos_nomes
        })

    if processos_lista:
        df = pd.DataFrame(processos_lista)
        # 'quoting=1' para garantir que o CSV envolva os campos em aspas
        df.to_csv(f"processos_ambientais_{ano}.csv", index=False, encoding='utf-8-sig', quoting=1)
        print(f"Arquivo de {ano} salvo com sucesso.")

In [4]:
for ano_atual in range(1985, 2026):
    print(f"Buscando: {ano_atual}...")
    resultado = buscar_periodo(ano_atual)
    salvar_csv(resultado, ano_atual)
    time.sleep(1) # Para evitar bloqueio

Buscando: 1985...
Total encontrado em 1985: 1
Arquivo de 1985 salvo com sucesso.
Buscando: 1986...
Total encontrado em 1986: 1
Arquivo de 1986 salvo com sucesso.
Buscando: 1987...
Total encontrado em 1987: 3
Arquivo de 1987 salvo com sucesso.
Buscando: 1988...
Total encontrado em 1988: 0
Buscando: 1989...
Total encontrado em 1989: 2
Arquivo de 1989 salvo com sucesso.
Buscando: 1990...
Total encontrado em 1990: 2
Arquivo de 1990 salvo com sucesso.
Buscando: 1991...
Total encontrado em 1991: 2
Arquivo de 1991 salvo com sucesso.
Buscando: 1992...
Total encontrado em 1992: 3
Arquivo de 1992 salvo com sucesso.
Buscando: 1993...
Total encontrado em 1993: 3
Arquivo de 1993 salvo com sucesso.
Buscando: 1994...
Total encontrado em 1994: 4
Arquivo de 1994 salvo com sucesso.
Buscando: 1995...
Total encontrado em 1995: 9
Arquivo de 1995 salvo com sucesso.
Buscando: 1996...
Total encontrado em 1996: 9
Arquivo de 1996 salvo com sucesso.
Buscando: 1997...
Total encontrado em 1997: 11
Arquivo de 1997 

Salvar todos os arquivos em um .csv único

In [5]:
arquivos = glob.glob("processos_ambientais_*.csv")

df_total = pd.concat([pd.read_csv(f) for f in arquivos], ignore_index=True)
df_total = df_total.drop_duplicates(subset=['numero'])
df_total.to_csv("PROCESSOS_AMBIENTAIS_TRF1_1985_2025.csv", index=False, encoding='utf-8-sig')

print(f"Concluído! Total de processos ambientais no DataJud: {len(df_total)}")

Concluído! Total de processos ambientais no DataJud: 58043
